# Phase 5 - Streamlit + LangGraph-style Clinician Tool

**Intended use:** Analytics decision-support only - not a medical device.

**LLM:** Ollama Desktop - **DeepSeek R1** (primary) → **Llama 3** (fallback) → deterministic template.

**Launch UI:** `streamlit run app_streamlit.py`

**Prerequisite:** Phase 3 champion artifacts and Phase 1 marts/scripts.


## 1. Setup


In [1]:
from __future__ import annotations
import json, hashlib, os, re, uuid, warnings, shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path(".").resolve()
if not (ROOT / "datafile.txt").exists():
    ROOT = Path("..").resolve()
os.chdir(ROOT)
DATAFILE = ROOT / "datafile.txt"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def load_registry(path=DATAFILE):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("|")
        if len(parts) < 4:
            continue
        rows.append({"role": parts[0].strip(), "zone": parts[1].strip(), "path": parts[2].strip(), "description": parts[3].strip()})
    return pd.DataFrame(rows)

def registry_paths(role=None):
    reg = load_registry()
    if role is not None:
        reg = reg[reg["role"] == role]
    return [ROOT / p for p in reg["path"].tolist()]

def registry_path(role, default=None):
    paths = registry_paths(role=role)
    if paths:
        return paths[0]
    return ROOT / default if default else None

def upsert_registry(role, zone, rel_path, description):
    lines = DATAFILE.read_text(encoding="utf-8").splitlines()
    new_line = f"{role}|{zone}|{rel_path}|{description}"
    out, found = [], False
    for line in lines:
        if line.startswith("#") or not line.strip():
            out.append(line)
            continue
        parts = line.split("|")
        if len(parts) >= 3 and parts[0].strip() == role and parts[2].strip() == rel_path:
            out.append(new_line)
            found = True
        else:
            out.append(line)
    if not found:
        out.append(new_line)
    DATAFILE.write_text("\n".join(out) + "\n", encoding="utf-8")

def new_run_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

def file_checksum(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("ROOT:", ROOT)
print(load_registry())


ROOT: E:\Amit\Project\Hospital project
           role    zone                                       path  \
0           raw  bronze                 data/raw/diabetic_data.csv   
1         clean  silver        data/lake/silver/encounters.parquet   
2      features    gold      data/lake/gold/model_features.parquet   
3     sequences    gold       data/lake/gold/rnn_sequences.parquet   
4   predictions    gold    data/lake/gold/test_predictions.parquet   
5       metrics    gold  data/lake/gold/experiment_results.parquet   
6          mart  export          data/exports/mart_readmission.csv   
7          mart  export        data/exports/mart_clinical_risk.csv   
8          mart  export    data/exports/mart_model_performance.csv   
9          mart  export         data/exports/mart_dq_scorecard.csv   
10     manifest     ops    data/lake/bronze/_manifests/latest.json   
11     champion     ops              models/champion_register.json   

                                      description 

## 2. Load champion pipeline and chatbot scripts

Serving model, feature list, gold-standard / FAQ JSON packs.


In [2]:
import joblib
import requests

register = json.loads((ROOT / "models" / "champion_register.json").read_text(encoding="utf-8"))
pipe = joblib.load(ROOT / "models" / "champion_pipeline.joblib")
feat_info = json.loads((ROOT / "models" / "champion_features.json").read_text(encoding="utf-8"))
FEATURE_COLS = feat_info["features"]

script_dir = ROOT / "chatbot" / "scripts"
SCRIPTS = []
for p in script_dir.glob("*.json"):
    SCRIPTS.extend(json.loads(p.read_text(encoding="utf-8")))
print("Champion:", register.get("champion_model"), "threshold:", register.get("threshold"))
print("Scripts loaded:", len(SCRIPTS), "features:", len(FEATURE_COLS))


Champion: rf threshold: 0.44999999999999996
Scripts loaded: 11 features: 25


## 3. Risk helpers

Risk bands (Low/Medium/High), rule-based recommendations, template explanation.


In [3]:
def risk_band(p):
    if p < 0.33:
        return "Low"
    if p < 0.66:
        return "Medium"
    return "High"

def rule_recommendations(band, row):
    recs = []
    if band == "High":
        recs.append("Follow-up check recommended")
        recs.append("Medication review needed")
    elif band == "Medium":
        recs.append("Follow-up check recommended")
    if float(row.get("total_visits", 0) or 0) >= 3:
        recs.append("Review frequent utilization pattern")
    if float(row.get("active_med_count", 0) or 0) >= 5:
        recs.append("Medication review needed")
    return recs or ["Continue routine monitoring per care pathway"]

def template_explanation(band, top_factors, recs):
    factors = ", ".join(top_factors[:3]) if top_factors else "documented risk factors"
    rec = recs[0] if recs else "Monitoring is recommended"
    return (
        f"This patient has a {band.lower()} risk of readmission due to {factors}. {rec}. "
        "Analytics decision-support only - not a medical device."
    )

print(risk_band(0.2), risk_band(0.5), risk_band(0.8))


Low Medium High


## 4. Script matcher (gold-standard / FAQ)

Deterministic answers from JSON script packs - no hallucination.


In [4]:
def match_script(message: str):
    msg = message.lower()
    best = None
    best_hits = 0
    for item in SCRIPTS:
        hits = sum(1 for p in item.get("patterns", []) if p.lower() in msg)
        if hits > best_hits:
            best_hits = hits
            best = item
    return best if best_hits > 0 else None

print(match_script("What is the primary outcome?"))


{'id': 'gs1', 'patterns': ['what is the target', 'primary outcome', '30 day', '30-day'], 'answer': 'Primary outcome is 30-day readmission (label <30). Recall is prioritized over accuracy because missing a high-risk patient is costly.'}


## 5. Semantic metrics (allowlisted)

Answers KPI questions from certified `mart_readmission` using the metric dictionary.


In [5]:
def semantic_metric(message: str):
    msg = message.lower()
    mart = pd.read_csv(ROOT / "data" / "exports" / "mart_readmission.csv")
    if "readmission rate" in msg or ("readmission" in msg and "rate" in msg):
        if "age" in msg:
            g = mart.groupby("age")["readmit_30d"].mean().sort_index()
            return "Readmission rate by age (metric dictionary): " + ", ".join([f"{i}={v:.3f}" for i, v in g.items()])
        if "gender" in msg:
            g = mart.groupby("gender")["readmit_30d"].mean()
            return "Readmission rate by gender: " + ", ".join([f"{i}={v:.3f}" for i, v in g.items()])
        return f"Overall 30d readmission rate = {mart['readmit_30d'].mean():.3f} (sum(readmit_30d)/count(encounters))."
    if "length of stay" in msg or "avg los" in msg or "average los" in msg:
        return f"Average LOS = {mart['time_in_hospital'].mean():.2f} days."
    return None

print(semantic_metric("What is the readmission rate?"))


Overall 30d readmission rate = 0.112 (sum(readmit_30d)/count(encounters)).


## 6. RAG over project documents

Retrieves grounded snippets from `rag_documents.json` (and Chroma when available).


In [6]:
def rag_answer(message: str):
    docs = json.loads((ROOT / "data" / "nosql" / "rag_documents.json").read_text(encoding="utf-8"))
    msg = message.lower().split()
    scored = []
    for d in docs:
        score = sum(1 for w in msg if w in d["text"].lower())
        scored.append((score, d))
    scored.sort(reverse=True, key=lambda x: x[0])
    if scored and scored[0][0] > 0:
        d = scored[0][1]
        return f"According to [{d['id']}]: {d['text']}"
    return None

print(rag_answer("Is this a medical device?"))


According to [intended_use]: Analytics decision-support only. Not a medical device. Not for standalone clinical decisions.


## 7. Ollama phrasing (DeepSeek R1 → Llama 3 → template)

LLM may only rewrite **provided facts**. Guardrails reject unsupported clinical claims.


In [7]:
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
PRIMARY_MODEL = os.environ.get("OLLAMA_PRIMARY", "deepseek-r1")
FALLBACK_MODEL = os.environ.get("OLLAMA_FALLBACK", "llama3")

def ollama_phrase(facts: dict):
    prompt = (
        "Use ONLY these facts. Do not add clinical advice beyond recommendations. "
        "If facts are insufficient, say so.\nFacts:\n" + json.dumps(facts, indent=2) +
        "\nWrite 1-2 clinician-facing sentences."
    )
    for model in [PRIMARY_MODEL, FALLBACK_MODEL]:
        try:
            r = requests.post(
                f"{OLLAMA_URL}/api/generate",
                json={"model": model, "prompt": prompt, "stream": False},
                timeout=60,
            )
            if r.status_code == 200:
                text = r.json().get("response", "").strip()
                if text:
                    banned = ["you should prescribe", "i diagnose", "definitely has cancer"]
                    if any(b in text.lower() for b in banned):
                        continue
                    return text, model
        except Exception as e:
            print("ollama fail", model, e)
            continue
    return None, None

print("Primary:", PRIMARY_MODEL, "Fallback:", FALLBACK_MODEL)


Primary: deepseek-r1 Fallback: llama3


## 8. `predict_row` - score one encounter

Probability, risk band, top factors, recommendations, grounded explanation.


In [8]:
def predict_row(row: dict):
    X = pd.DataFrame([{c: row.get(c, np.nan) for c in FEATURE_COLS}])
    prob = float(pipe.predict_proba(X)[0, 1])
    thr = float(register.get("threshold", 0.5))
    band = risk_band(prob)
    top = [f.get("feature", "") for f in register.get("top_features", [])[:5]]
    recs = rule_recommendations(band, row)
    facts = {
        "risk_band": band,
        "probability": round(prob, 4),
        "threshold": thr,
        "top_factors": top,
        "recommendations": recs,
        "model_id": register.get("champion_model"),
    }
    text, model_id = ollama_phrase(facts)
    if not text:
        text = template_explanation(band, top, recs)
        model_id = "template"
    return {**facts, "explanation": text, "llm_model": model_id, "alert": band == "High"}


## 9. LangGraph-style router

Flow: RBAC → input guardrails → script_qa | semantic metrics | RAG | refuse → audit log.


In [9]:
def route_message(message: str, role: str = "clinician"):
    rbac = json.loads((ROOT / "data" / "nosql" / "rbac_roles.json").read_text(encoding="utf-8"))
    role_perm = rbac["roles"].get(role, rbac["roles"]["viewer"])
    audit_path = ROOT / "data" / "nosql" / "audit_events.json"
    audit = json.loads(audit_path.read_text(encoding="utf-8"))

    lowered = message.lower()
    if any(x in lowered for x in ["prescribe", "diagnose me", "what drug should"]):
        ans = match_script(message)
        out = ans["answer"] if ans else "I cannot provide medical diagnosis or prescribing advice."
        audit.append({"role": role, "action": "refuse_medical_advice", "ts": datetime.now(timezone.utc).isoformat()})
        audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
        return {"route": "refuse", "answer": out}

    hit = match_script(message)
    if hit:
        audit.append({"role": role, "action": "script_qa", "id": hit["id"], "ts": datetime.now(timezone.utc).isoformat()})
        audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
        return {"route": "script_qa", "answer": hit["answer"]}

    sem = semantic_metric(message)
    if sem:
        if role == "viewer" and "patient" in lowered:
            return {"route": "refuse", "answer": "Viewer role cannot access patient-level detail."}
        audit.append({"role": role, "action": "semantic_sql", "ts": datetime.now(timezone.utc).isoformat()})
        audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
        return {"route": "semantic_metric_sql", "answer": sem}

    rag = rag_answer(message)
    if rag:
        audit.append({"role": role, "action": "rag", "ts": datetime.now(timezone.utc).isoformat()})
        audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
        return {"route": "vector_rag", "answer": rag}

    audit.append({"role": role, "action": "refuse_ungrounded", "ts": datetime.now(timezone.utc).isoformat()})
    audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
    return {"route": "refuse", "answer": "I can only answer from project knowledge, gold-standard scripts, certified metrics, or a structured risk prediction."}


## 10. Smoke tests

Predict one row and route sample chat questions.


In [10]:
sample = pd.read_parquet(registry_path("features", "data/lake/gold/model_features.parquet")).iloc[0].to_dict()
pred = predict_row(sample)
print("Predict smoke:", {k: pred[k] for k in ["risk_band", "probability", "llm_model", "alert"]})
print("Explanation:", pred["explanation"][:200])
print("Chat:", route_message("What is the primary outcome?"))
print("Chat:", route_message("What is the readmission rate?"))


ollama fail deepseek-r1 HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)


Predict smoke: {'risk_band': 'Low', 'probability': 0.2464, 'llm_model': 'llama3', 'alert': False}
Explanation: Based on the provided facts, here are two clinician-facing sentences:

According to the model's assessment, this patient is classified as low-risk. As such, I recommend continuing routine monitoring p
Chat: {'route': 'script_qa', 'answer': 'Primary outcome is 30-day readmission (label <30). Recall is prioritized over accuracy because missing a high-risk patient is costly.'}


Chat: {'route': 'semantic_metric_sql', 'answer': 'Overall 30d readmission rate = 0.112 (sum(readmit_30d)/count(encounters)).'}


## 10b. LangGraph MCP router (runtime)

Real `StateGraph` router using the shared MCP service pool (`mcp/client/pool.py`).
Same tools as Cursor MCP servers, imported directly for low latency in notebooks.

In [ ]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from typing import TypedDict
from langgraph.graph import StateGraph, END
from mcp.client.pool import pool

class ChatState(TypedDict):
    message: str
    role: str
    route: str
    answer: str

def guard_node(state: ChatState) -> ChatState:
    msg = state["message"].lower()
    if any(x in msg for x in ["prescribe", "diagnose me", "what drug should"]):
        return {**state, "route": "refuse", "answer": "I cannot provide medical diagnosis or prescribing advice."}
    return state

def route_node(state: ChatState) -> ChatState:
    if state.get("route") == "refuse" and state.get("answer"):
        return state
    msg = state["message"]
    lowered = msg.lower()
    if "fred" in lowered or "unemployment" in lowered:
        sid = "UNRATE" if "unemployment" in lowered else "CPIAUCSL"
        return {**state, "route": "fred_mcp", "answer": str(pool.fred_series(sid))}
    hit = match_script(msg)
    if hit:
        return {**state, "route": "script_qa", "answer": hit["answer"]}
    sem = pool.semantic_metric(msg)
    if sem:
        return {**state, "route": "semantic_metric_mcp", "answer": sem}
    if "select" in lowered and "from" in lowered:
        return {**state, "route": "sqlite_mcp", "answer": pool.sqlite_query(msg)}
    rag = pool.rag_answer(msg)
    if rag:
        return {**state, "route": "vector_rag_mcp", "answer": rag}
    return {**state, "route": "refuse", "answer": "I can only answer from project knowledge, scripts, or certified metrics."}

def audit_node(state: ChatState) -> ChatState:
    pool.audit(state["role"], state["route"])
    return state

def build_mcp_graph():
    g = StateGraph(ChatState)
    g.add_node("guard", guard_node)
    g.add_node("route", route_node)
    g.add_node("audit", audit_node)
    g.set_entry_point("guard")
    g.add_edge("guard", "route")
    g.add_edge("route", "audit")
    g.add_edge("audit", END)
    return g.compile()

mcp_graph = build_mcp_graph()

def route_message_mcp(message: str, role: str = "clinician") -> dict:
    out = mcp_graph.invoke({"message": message, "role": role, "route": "", "answer": ""})
    return {"route": out["route"], "answer": out["answer"]}

print("MCP pool health:", pool.health_summary())
print("LangGraph MCP:", route_message_mcp("What is the readmission rate?"))
print("LangGraph MCP RAG:", route_message_mcp("What is the intended use of this model?"))

## 11. Launch Streamlit

The interactive UI lives in `app_streamlit.py` (only allowed thin `.py` entrypoint).


In [11]:
print("Phase 5 assets validated.")
print("Launch: streamlit run app_streamlit.py")
print("Ollama models: ollama pull deepseek-r1 && ollama pull llama3")


Phase 5 assets validated.
Launch: streamlit run app_streamlit.py
Ollama models: ollama pull deepseek-r1 && ollama pull llama3


## 10c. MCP Model Tribunal (multi-gate LangGraph)

Governance stakeholder gates before clinician-facing chat: clinical guard → config → tool router → fairness → audit.

In [ ]:
from inference.tribunal import (
    clinical_guard,
    config_gate,
    tool_router,
    fairness_gate,
    audit_node,
    route_message_tribunal,
)

class TribunalState(TypedDict):
    message: str
    role: str
    route: str
    answer: str
    stages: list
    fairness_warning: str

def build_tribunal_graph():
    g = StateGraph(TribunalState)
    g.add_node("clinical_guard", clinical_guard)
    g.add_node("config_gate", config_gate)
    g.add_node("tool_router", lambda s: tool_router(s, match_script))
    g.add_node("fairness_gate", fairness_gate)
    g.add_node("audit", audit_node)
    g.set_entry_point("clinical_guard")
    g.add_edge("clinical_guard", "config_gate")
    g.add_conditional_edges(
        "config_gate",
        lambda s: "end" if s.get("route") == "refuse" else "route",
        {"end": END, "route": "tool_router"},
    )
    g.add_edge("tool_router", "fairness_gate")
    g.add_edge("fairness_gate", "audit")
    g.add_edge("audit", END)
    return g.compile()

tribunal_graph = build_tribunal_graph()

def route_message_tribunal_graph(message: str, role: str = "clinician") -> dict:
    out = tribunal_graph.invoke({
        "message": message,
        "role": role,
        "route": "",
        "answer": "",
        "stages": [],
        "fairness_warning": "",
    })
    return {"route": out["route"], "answer": out["answer"], "stages": out.get("stages", [])}

print("Tribunal smoke:", route_message_tribunal_graph("What is the readmission rate?"))
print("Tribunal refuse:", route_message_tribunal_graph("prescribe me antibiotics"))
print("Tribunal flat:", route_message_tribunal("What is the primary outcome?", "clinician", match_script))